In [10]:
!pip install -q -U langchain langchain-openai langchain-tavily python-dotenv


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from dotenv import load_dotenv
import os
import json

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

load_dotenv()

print("OpenAI:", os.getenv("OPENAI_API_KEY") is not None)
print("Tavily:", os.getenv("TAVILY_API_KEY") is not None)

OpenAI: True
Tavily: True


In [12]:
llm = ChatOpenAI(
    model="gpt-5.6-luna",
    use_responses_api=True
)

print("LLM ready")

LLM ready


In [13]:
with open("thebayrestaurant_jed.json", "r", encoding="utf-8") as f:
    evidence = json.load(f)

evidence_text = json.dumps(
    evidence,
    indent=2,
    ensure_ascii=False
)

print("Research Evidence loaded")
print(evidence_text[:1000])

Research Evidence loaded
{
  "restaurant": {
    "restaurant_id": 3,
    "name": "The Bay Restaurant",
    "instagram_username": "thebayrestaurant_jed",
    "instagram_url": "https://www.instagram.com/thebayrestaurant_jed/",
    "email": null,
    "location": "Jedaah",
    "category": "Cafe, Restaurant, Food & Beverage Company"
  },
  "analysis_metadata": {
    "status": "partial",
    "analyzed_at": "2026-09-14T07:41:51.804323+00:00",
    "posts_scraped": 6,
    "posts_analyzed": 5,
    "posts_failed": 1,
    "source": "instagram",
    "analysis_version": "1.0"
  },
  "profile": {
    "username": "thebayrestaurant_jed",
    "full_name": "•The BAY• 🍃 •ذا باي•",
    "bio": "Refined Indian Cuisine 🇮🇳 \nBold Flavors, Modern Soul.\n📍The bay, Jeddah",
    "followers": 20474,
    "following": 1,
    "posts_count": 432,
    "website": "https://linktr.ee/thebayrestaurant_jed?utm_source=linktree_profile_share&ltsid=e5bd1dd7-599f-4005-84d2-9819fd8d0f9e",
    "verified": false,
    "business_cate

In [28]:
web_search = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced"
)


@tool
def search_instagram_benchmark(query: str) -> str:
    """
    Search the web for relevant and current Instagram
    marketing benchmarks.

    Use this tool when an external benchmark is needed
    to evaluate a restaurant's Instagram performance.

    Do not use this tool to retrieve the restaurant's
    own Instagram data.

    The results may contain external sources, benchmarks,
    and industry findings. Verify relevance before using
    them in the qualification report.
    """

    results = web_search.invoke({
        "query": query
    })

    return str(results)
print("Benchmark Tool ready")

Benchmark Tool ready


In [29]:
test_result = search_instagram_benchmark.invoke(
    "2026 Instagram posting frequency engagement benchmark for restaurants"
)

print(test_result)

{'query': '2026 Instagram posting frequency engagement benchmark for restaurants', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://evokad.com/restaurant-social-media-marketing-guide-2026', 'title': 'The Restaurant Social Media Marketing Guide 2026', 'content': 'For restaurants specifically, the platform picture gets more nuanced. Hootsuite’s industry benchmarks place dining and hospitality brands at around 3.1% engagement on Instagram, well above the cross-industry average. That premium exists because food content performs natively on visual platforms. What most CMOs miss is how much the numbers shift when follower size is factored in. Accounts with fewer than 10,000 followers average 4.7% engagement on TikTok and 2.8% on Instagram, significantly higher than those with more than 100,000 followers. [...] The highest-converting Instagram content mix for restaurants combines Reels for reach, carousels for depth, and Stories for reservation prompts. 

In [27]:
qualification_prompt = """
You are the Qualification and Marketing Gap Analysis Agent
for Rawaj, an Agentic AI System for Restaurant Marketing.

Your role is to analyze research evidence about restaurants,
identify evidence-based Instagram marketing gaps, and qualify
restaurants as potential marketing leads.

You are an ANALYST, not a strategy or outreach agent.

==================================================
CORE RESPONSIBILITIES
==================================================

1. Analyze the restaurant's research evidence.
2. Identify relevant Instagram marketing gaps.
3. Use external benchmarks when a comparison is needed.
4. Compare the restaurant's performance with relevant benchmarks.
5. Support every finding with clear evidence.
6. Qualify the restaurant based on the available evidence.
7. Assign severity and priority to each confirmed gap.
8. Clearly report missing or insufficient data.

Do not create marketing strategies.
Do not generate outreach messages.
Do not estimate revenue impact.
Do not invent facts, metrics, sources, or benchmarks.

==================================================
EVIDENCE AND HALLUCINATION RULES
==================================================

1. Use the provided research evidence as the primary source.

2. Do not invent or assume:
   - Engagement rates
   - Follower counts
   - Posting frequency
   - Content types
   - CTA usage
   - Pricing information
   - Website features
   - External benchmark values
   - Sources or URLs

3. When an external benchmark is required:
   - Use the search_instagram_benchmark tool first.
   - Do not rely on remembered benchmark values.
   - Use only relevant and credible search findings.
   - Include the source or URL when available.
   - Explain how the benchmark relates to the restaurant.

4. Do not use external benchmarks if they are not
   relevant or cannot be verified.

5. Clearly distinguish between:
   - Direct evidence from the research.
   - External benchmark findings.
   - Inferences based on the evidence.
   - Missing or unavailable information.

6. If a benchmark is not available or cannot be verified,
   state the limitation instead of inventing a value.

==================================================
INCOMPLETE DATA RULES
==================================================

1. Missing information does not prove that a feature
   or activity does not exist.

2. Never claim that a restaurant has no CTA, no pricing,
   or no website feature unless the relevant information
   was sufficiently inspected.

3. If a CTA was not found in the available sample, write:

   "CTA was not observed in the provided sample."

4. Do not write:

   "The restaurant has no CTA."

5. Use the following labels when appropriate:

   - Confirmed: Supported by direct evidence.
   - Not observed: Not found in the inspected sample.
   - Not available: The required data was not provided.
   - Uncertain: Evidence is insufficient to confirm the claim.

6. Do not classify a missing data point as a confirmed
   marketing gap.

7. Mention relevant data limitations in the final report.

==================================================
MARKETING GAP IDENTIFICATION
==================================================

Identify a marketing gap only when:

1. There is supporting evidence.
2. The issue is relevant to restaurant marketing.
3. The available data is sufficient for the conclusion.
4. The conclusion is not based only on missing information.

For every marketing gap, include:

- Gap name
- Description
- Severity: High, Medium, or Low
- Priority: 1 is the highest priority
- Confidence: High, Medium, or Low
- Supporting evidence
- Benchmark evidence, if used
- Explanation of why the evidence indicates a gap

If the evidence is insufficient, do not create a confirmed gap.
Report the issue under data limitations instead.

==================================================
BENCHMARK SEARCH RULES
==================================================

Use the search_instagram_benchmark tool when:

- An external benchmark is needed for comparison.
- A conclusion depends on industry standards.
- The provided evidence does not contain the required benchmark.

Use specific search queries such as:

- Restaurant Instagram engagement benchmarks
- Restaurant Instagram posting frequency benchmarks
- Instagram CTA best practices for restaurants
- Restaurant Instagram content performance benchmarks

After searching:

1. Check the relevance of the results.
2. Avoid unsupported numerical claims.
3. Do not treat every search result as reliable evidence.
4. Do not apply a benchmark directly if the populations
   or measurement methods are not comparable.
5. Clearly mention limitations when the comparison
   is not directly applicable.

==================================================
QUALIFICATION RULES
==================================================

The qualification decision must be based on:

1. The quality and completeness of the evidence.
2. The relevance and severity of identified marketing gaps.
3. The restaurant's observed marketing needs.
4. The confidence level of the findings.

Do not qualify a restaurant based on assumptions.
If the evidence is insufficient, state that the decision
has limited confidence and explain why.

==================================================
FINAL REPORT REQUIREMENTS
==================================================

Return a clear and organized qualification report
containing:

1. Restaurant information.
2. Qualification decision.
3. Decision rationale.
4. Marketing gaps.
5. Severity and priority for each gap.
6. Supporting evidence.
7. External benchmark evidence, if used.
8. Strengths.
9. Data limitations.

Before finalizing, check:

- Is every gap supported by evidence?
- Did I distinguish missing data from confirmed absence?
- Did I use the benchmark tool when required?
- Did I avoid inventing metrics or sources?
- Did I avoid creating strategies or outreach messages?
- Did I clearly state uncertainty?
"""

In [17]:
from langchain.agents import create_agent

tools = [search_instagram_benchmark]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=qualification_prompt
)

print("Agent ready")

Agent ready


In [ ]:
import json

research_evidence = json.dumps(
    evidence,
    ensure_ascii=False,
    indent=2
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""
Analyze the following restaurant research evidence.

Research Evidence:
{research_evidence}

Follow the system instructions strictly.

Important:
- Use the provided evidence as the primary source.
- Use the benchmark search tool when an external
  comparison is required.
- Do not assume missing information means absence.
- Clearly distinguish confirmed gaps from data limitations.
- Do not invent metrics, sources, or benchmark values.
- Return a complete qualification report.
"""
        }
    ]
})

In [19]:
for i, message in enumerate(result["messages"]):
    print(f"\n===== MESSAGE {i} =====")
    print("TYPE:", type(message).__name__)

    if type(message).__name__ == "AIMessage":
        print("CONTENT:")
        print(message.content)

        print("\nTEXT:")
        print(message.text)

    elif type(message).__name__ == "ToolMessage":
        print("TOOL:", message.name)


===== MESSAGE 0 =====
TYPE: HumanMessage

===== MESSAGE 1 =====
TYPE: AIMessage
CONTENT:
[{'id': 'rs_04682807e30530d2006aaadcb243b487d2883308682cf7fa37', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqqty0hSI4_XS_EWuiCvB0ExqFqbO3y30H8l99GKGB-6GwHQPKCzgAkrVPkjiXqdYXpGpuXAvZY2QXTZkXPMDeHNvEcT3gygWmlGetL3WLWj-MdXzfi_gsVf-N1LsJ1gvZzClZWhpOK-Xn6f-87LdiP9bhTheqLx40HFI3t9cAqFe8_4yp0oQr2LSyi9GbqlBYLc-UD17kVtVoKLNxXgoxA8TBXxnt3vO51zf8dK_Eu3zEqWyziKMW9M0GWqhH4bOauIFX0Zp7PR72wWZJI7DS4VtkbkcQ_1g1N3JNmaY6Oh2KhRflX8NkE6nIfvgNiufZmTtfpnXwUvZ-lU9JXDXf2D9Rqp-zvphSroWLB4lGVloDr7b3s-USf7RsL4oeRb-RKKB7jXxq6VS6bgEvU7l0XRnSbAdSG9UmJTznoBiLSh8TzMPIF8bsnV6g1mLdHyaoGWznFTO1eayG-IK-_YKpIVdl1c5mSW8UN8b-UDpcdMrmT0JZZuDcAf7VCyKU87N3SQrlM00aXwJ-w39w9P_gisnkF-xdKU3NRmRfubmc1NKA517bdNQZA1lLPI2SmIDo2Mf7woAaiV52T7xqv3AUBqcydzsCZMwXII2vER_nWwt2gEDETci6Ed9Aw4_fZ-HORViRhOwKIKRbPgzn36lhHvbJ389xXWgfIz1jWmj5x1QumHzA5LIx8mD21DrwwdyFlSXyjOG4QjVESFxydzy0nkmZ6KWrQWxcY5Sq3uKyepLy8A6AwAIEBmEUKwi4

In [30]:
import json

from pydantic import BaseModel, Field
from typing import List


class Gap(BaseModel):
    gap: str
    severity: str
    priority: int
    evidence: List[str] = Field(default_factory=list)
    recommendation_focus: str


class Report(BaseModel):
    restaurant: str
    qualification: str
    decision_rationale: str
    marketing_gaps: List[Gap] = Field(default_factory=list)
    strengths: List[str] = Field(default_factory=list)
    data_limitations: List[str] = Field(default_factory=list)


# Convert the qualification output into structured JSON
structured_llm = llm.with_structured_output(Report)

structured_result = structured_llm.invoke(f"""
Convert the following qualification report into
organized structured JSON.

Rules:
- Extract information only from the report.
- Do not invent data.
- Keep each marketing gap separate.
- Preserve evidence and limitations.

Qualification Report:
{final_report}
""")


# Save the organized JSON file
output = structured_result.model_dump()

output["restaurant_id"] = evidence.get("restaurant_id")
output["agent"] = "Qualification & Marketing Gap Analysis Agent"

with open(
    "thebayrestaurant_qualification.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Qualification JSON saved successfully.")

Qualification JSON saved successfully.


# if any one did not like lines of reading this is the summery from the agent hahahah 

In [31]:
%pip install -U langsmith openevals

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
def qualification_target(inputs: dict) -> dict:
    evidence = inputs["evidence"]

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"""
Analyze this restaurant research evidence.

Research Evidence:
{evidence}

Identify marketing gaps and qualify the restaurant.
Use evidence only and do not invent data.
"""
            }
        ]
    })

    final_message = result["messages"][-1]

    if isinstance(final_message.content, list):
        report = "".join(
            item.get("text", "")
            for item in final_message.content
            if isinstance(item, dict)
            and item.get("type") == "text"
        )
    else:
        report = str(final_message.content)

    return {
        "qualification_report": report
    }

In [34]:
from langsmith import Client
import json

client = Client()

dataset_name = "Rawaj Qualification Evaluation"

# Get existing dataset
dataset = client.read_dataset(dataset_name=dataset_name)

# Add a new example
client.create_example(
    inputs={
        "evidence": json.dumps(
            evidence,
            ensure_ascii=False,
            indent=2
        )
    },
    dataset_id=dataset.id
)

print("Example added to existing dataset:", dataset.name)

Example added to existing dataset: Rawaj Qualification Evaluation


In [38]:
from openevals.llm import create_llm_as_judge

evaluator = create_llm_as_judge(
    prompt="""
You are evaluating a Qualification and Marketing Gap
Analysis Agent for a restaurant marketing platform.

Evaluate the agent's output using these criteria:

1. Evidence Grounding:
Are the marketing gaps supported by research evidence?

2. Qualification Decision:
Is the qualification decision reasonable?

3. Gap Identification:
Are the identified marketing gaps relevant?

4. No Hallucination:
Did the agent avoid inventing metrics or facts?

5. Completeness:
Does the report include evidence, severity,
priority, and limitations when available?

Give a score from 0 to 1:
- 1.0 = Excellent
- 0.5 = Partially correct
- 0.0 = Incorrect or unsupported

Explain your reasoning.

Inputs:
{inputs}

Agent Output:
{outputs}
""",
    model="openai:gpt-5.6-luna",
    continuous=True
)

In [39]:
results = client.evaluate(
    qualification_target,
    data=dataset.name,
    evaluators=[
        evaluator
    ],
    experiment_prefix="qualification-agent-test",
    description="Evaluation of Rawaj Qualification Agent",
    max_concurrency=1
)

print("Evaluation completed.")

View the evaluation results for experiment: 'qualification-agent-test-0dece739' at:
https://smith.langchain.com/o/2b1b51ad-d66a-4eea-81d3-fdc51ca7d380/datasets/f9406d0c-fd98-4e30-a494-3c56ab0e61a3/compare?selectedSessions=6b1618da-ef11-41bd-bb1b-898a97990e0f




0it [00:00, ?it/s]

Evaluation completed.
